## 1. What a test does

A test calls code with a controlled input and checks an observable result. It should answer one specific question.

```text
Arrange input -> Act by calling code -> Assert the result
```

`pytest` discovers functions whose names begin with `test_` inside files normally named `test_*.py`.

### Completed example using unrelated code

```python
def shipping_cost(order_total: float) -> float:
    if order_total >= 50:
        return 0.0
    return 5.0


def test_shipping_is_free_at_boundary() -> None:
    # Arrange
    order_total = 50.0

    # Act
    result = shipping_cost(order_total)

    # Assert
    assert result == 0.0
```

`assert condition` returns nothing when the condition is true. When false, it raises `AssertionError`. Pytest catches that error and reports the test as failed.

### Test syntax anatomy

```python
def test_behavior_being_checked() -> None:
    actual = function_under_test(input_value)
    assert actual == expected_value
```

| Part | Meaning |
|---|---|
| `test_...` | Makes the function discoverable by pytest |
| `-> None` | The test does not return a useful result |
| `actual` | The result produced by your code |
| `expected_value` | The result required by the contract |
| `assert` | Fail this test if the condition is false |

Tests do not need to call `print()`. A printed result still requires a human to inspect it. An assertion makes the check automatic.

## 2. Add pytest to the project

Pytest is not currently installed in your MLS environment. Declare it as a development dependency instead of making it a runtime requirement.

Add this to the project-root `pyproject.toml`:

```toml
[project.optional-dependencies]
dev = ["pytest>=8,<9"]
```

Meaning:

- `optional-dependencies` groups tools not required by normal package users.
- `dev` is the group name chosen for development tools.
- `>=8,<9` accepts compatible pytest 8 releases while avoiding an unreviewed major upgrade.

From `Code/python_engineering`, run:

```powershell
python -m pip install -e ".[dev]"
python -m pytest --version
```

PowerShell quotes `.[dev]` so it reaches pip as one argument. The dot still means the current project.

### CHECKPOINT 1

Do these exact actions:

1. Confirm the MLS environment is active with `python -c "import sys; print(sys.executable)"`.
2. Add the development dependency.
3. Install the project with the `dev` extra.
4. Run `python -m pytest --version`.

Stop if the executable is not inside the MLS environment or pytest is unavailable.

## 3. Create the first real test file

From `Code/python_engineering`, create the folder and file yourself:

```powershell
New-Item -ItemType Directory -Path .\tests -Force
New-Item -ItemType File -Path .\tests\test_validation.py -Force
```

Write a test for the valid case of `validate_binary_label`.

### Syntax hint

```python
from ml_utils import name_to_test


def test_descriptive_behavior() -> None:
    result = name_to_test(valid_input)
    assert result is None
```

Use `is None`, not `== None`. The validator's successful contract is to finish without raising and return `None`.

Run only that file:

```powershell
python -m pytest tests/test_validation.py -v
```

Expected shape of output:

```text
collected 1 item
tests/test_validation.py::test_... PASSED
```

- `collected 1 item` means discovery found one test.
- The long name identifies the file and test function.
- `PASSED` means the function completed without an uncaught failure.

If pytest reports `collected 0 items`, check the test filename and function name first.

## 4. Testing expected exceptions with `pytest.raises`

An invalid input should raise `ValueError`. That error is expected behavior, so the test passes only when it occurs.

### Completed unrelated example

```python
import pytest


def set_volume(level: int) -> None:
    if level not in range(0, 11):
        raise ValueError("level must be from 0 to 10")


def test_volume_rejects_value_above_ten() -> None:
    with pytest.raises(ValueError, match="level must be"):
        set_volume(11)
```

Syntax flow:

```text
with pytest.raises(expected_error):
    call_that_should_raise()
```

`pytest.raises(...)` returns a context manager. It watches code inside the indented block. The optional `match` checks the error message using a regular-expression pattern.

### Your TODO

Add one test to `test_validation.py` proving that label `2` raises `ValueError`.

Syntax hint:

```python
import pytest

with pytest.raises(EXCEPTION_TYPE, match="stable part of message"):
    function_call(...)
```

Do not catch the error using your own `try/except` in this test. Let `pytest.raises` perform the assertion.

### Common failure output

```text
Failed: DID NOT RAISE <class 'ValueError'>
```

Meaning: the call completed normally even though the test required `ValueError`.

```text
AssertionError: Regex pattern did not match
```

Meaning: the expected exception type occurred, but its message did not match your `match` pattern. Prefer a short, stable part of the message rather than the entire sentence.

### CHECKPOINT 2

Run:

```powershell
python -m pytest tests/test_validation.py -v
```

Required result: two tests pass. One covers a valid label and one covers an invalid label.

## 5. Normal, boundary, and invalid cases

A useful test suite samples different parts of a function's contract.

| Case | Question | `TrainingConfig` example |
|---|---|---|
| Normal | Does ordinary valid input work? | learning rate `0.01` |
| Boundary | Does the exact allowed edge work? | threshold `0.0` or `1.0` |
| Invalid | Is forbidden input rejected? | learning rate `0.0` |

Boundary tests matter because mistakes often use `<` when they need `<=`, or the reverse.

Create `tests/test_config.py`. Write these tests yourself:

1. A normal valid configuration stores all three values.
2. Threshold `0.0` is accepted.
3. Threshold `1.0` is accepted.
4. Learning rate `0.0` raises `ValueError`.
5. Epochs `0` raises `ValueError`.
6. Threshold above `1.0` raises `ValueError`.

Use direct attribute assertions for valid instances and `pytest.raises` for invalid construction.

### Attribute-test syntax

```python
def test_object_stores_value() -> None:
    instance = ClassName(required_value, another_value)
    assert instance.attribute_name == expected_value
```

This checks observable behavior. Do not test dataclass implementation details such as generated internal methods.

## 6. Test the existing accuracy metric

Create `tests/test_metrics.py`. Test one behavior per function:

1. Perfect predictions return `1.0`.
2. Two correct predictions out of three return approximately two thirds.
3. Empty lists raise `ValueError`.
4. Unequal lengths raise `ValueError`.
5. An invalid true label raises `ValueError`.
6. An invalid predicted label raises `ValueError`.

### Floating-point syntax

Some decimal results cannot be represented exactly as binary floating-point numbers. Pytest provides an approximate comparison:

```python
assert actual_float == pytest.approx(expected_float)
```

`pytest.approx(...)` returns a comparison helper. It does not change your production value.

### CHECKPOINT 3

Run the complete suite:

```powershell
python -m pytest -q
```

`-q` means quiet output. It still shows failures and the final summary.

Before fixing a failure, read:

1. The final assertion or exception message.
2. The failing test name.
3. The source line marked by pytest.
4. The values shown under the failed assertion.

Do not change production code merely to make a wrong expectation pass. Decide whether the contract, implementation, or test is wrong.

## 7. Regression tests

A regression test preserves a bug fix. The sequence is:

```text
reproduce bug -> write failing test -> fix production code -> test passes forever
```

Your earlier `rang` typo caused `NameError` only when `calculate_accuracy()` reached the loop. A normal accuracy test now protects that execution path. If a similar undefined name returns, the suite fails automatically.

The test should describe behavior, not the old implementation mistake. Prefer `test_accuracy_counts_matching_labels` over `test_rang_typo_is_fixed`.

## 8. Capstone contract

Build a small evaluator that converts probabilities to labels and reports accuracy plus confusion counts.

Add these public interfaces:

```python
validate_probability(probability: float) -> None

confusion_counts(
    y_true: list[int],
    y_pred: list[int],
) -> dict[str, int]

evaluate_binary_classifier(
    y_true: list[int],
    probabilities: list[float],
    config: TrainingConfig,
) -> dict[str, float | int]
```

Return dictionary keys from the evaluator:

```text
accuracy, true_positive, true_negative, false_positive, false_negative
```

This is a learning interface. In a larger production project, a typed result model could be clearer than a mixed-value dictionary.

## 9. Implement one layer at a time

### Layer A: probability validation

Add `validate_probability()` to `validation.py`.

Contract:

- Accept values from `0.0` through `1.0`, inclusive.
- Raise `ValueError` outside that range.
- Return `None` when valid.

Syntax hint:

```python
def function_name(value: float) -> None:
    if LOWER_BOUND_CONDITION or UPPER_BOUND_CONDITION:
        raise ValueError("clear message")
```

Then add normal, boundary, and invalid tests before moving on.

### Layer B: confusion counts

Add `confusion_counts()` to `metrics.py`.

Mental model:

```text
actual 1, predicted 1 -> true positive
actual 0, predicted 0 -> true negative
actual 0, predicted 1 -> false positive
actual 1, predicted 0 -> false negative
```

Requirements:

- Reuse binary-label validation.
- Reject empty and unequal-length lists.
- Return all four keys even when a count is zero.
- Do not print.

Syntax hints:

```python
counts: dict[str, int] = {
    "true_positive": 0,
    # remaining keys
}

for actual, predicted in zip(y_true, y_pred):
    # compare the pair and increment exactly one count
```

`zip(first, second)` returns an iterator of paired items. Do not rely on `zip` to detect unequal lengths because it silently stops at the shorter list. Validate lengths first.

### Layer C: evaluator module

Create `src/ml_utils/evaluator.py` and define `evaluate_binary_classifier()`.

Processing flow:

```text
y_true + probabilities + config
    -> validate each probability
    -> threshold probabilities into predicted labels
    -> calculate accuracy
    -> calculate confusion counts
    -> combine results into one dictionary
```

Threshold rule for this lab:

```text
probability >= config.threshold -> 1
probability <  config.threshold -> 0
```

List-comprehension syntax hint:

```python
predictions = [VALUE_IF_TRUE if CONDITION else VALUE_IF_FALSE for item in items]
```

Dictionary-combination syntax hint:

```python
result: dict[str, float | int] = {"accuracy": accuracy}
result.update(counts)
return result
```

`dict.update(other)` mutates `result` and returns `None`. Do not write `return result.update(counts)`.

### Layer D: public exports

Update `ml_utils/__init__.py` to export:

- `validate_probability`
- `confusion_counts`
- `evaluate_binary_classifier`

Use the same absolute-import pattern from Session 4. Importing `ml_utils` must still produce no printed demonstration output.

## 10. Capstone tests

Create `tests/test_evaluator.py`. Required tests:

1. A normal example returns the expected accuracy and four counts.
2. A probability exactly equal to the threshold becomes label `1`.
3. An invalid probability raises `ValueError`.
4. Unequal `y_true` and probability lengths raise `ValueError`.
5. Repeating the same call returns equal dictionaries.

For the first test, manually calculate the expected labels and confusion counts on paper before writing assertions. Do not calculate the expected result by calling the same production helpers inside the test. That would repeat the implementation instead of independently checking it.

### Partially completed test shape

This shows structure, not the answer:

```python
from ml_utils import TrainingConfig, evaluate_binary_classifier


def test_evaluator_reports_expected_metrics() -> None:
    # Arrange
    config = TrainingConfig(learning_rate=..., epochs=..., threshold=...)
    y_true = [...]
    probabilities = [...]

    # Act
    result = evaluate_binary_classifier(y_true, probabilities, config)

    # Assert one key at a time
    assert result["accuracy"] == ...
    # TODO: assert all four confusion counts
```

Choose your own small data. Write the expected predictions on paper first.

## 11. Useful pytest commands

| Command | Purpose |
|---|---|
| `python -m pytest` | Run the full discovered suite |
| `python -m pytest -q` | Run with a compact summary |
| `python -m pytest -v` | Show every collected test name |
| `python -m pytest -x` | Stop after the first failure |
| `python -m pytest tests/test_metrics.py` | Run one file |
| `python -m pytest tests/test_metrics.py::test_name` | Run one test |

Use the full suite before declaring the session complete. A focused command helps while debugging, but it can miss breakage elsewhere.

## 12. Final acceptance check

Run from `Code/python_engineering`:

```powershell
python -m pytest -q
python -c "import ml_utils; print('clean import')"
python -m ml_utils
git status --short
git diff --check
```

Required outcomes:

- All tests pass.
- Import prints only `clean import`.
- The package command still works.
- Test names describe behavior.
- No test depends on execution order.
- No production function prints unexpectedly.
- `git diff --check` reports no whitespace errors.

Send me the pytest output and your files for critical review before committing.

## Final reflection: only three questions

Answer after all tests pass. Use one or two complete sentences each.

In [ ]:
final_reflection = {
    "How does a pytest test turn an expected behavior into an automatic check?": "",
    "When should a test use a normal assertion, and when should it use pytest.raises?": "",
    "Which capstone test would catch the most damaging realistic bug, and why?": "",
}

## Retrieval task: 2–3 days later

Without this notebook, rebuild one normal test, one boundary test, and one expected-exception test for a tiny function. Then run them with pytest. Record only the command and final result below.

In [ ]:
retrieval_date = ""
retrieval_command = ""
retrieval_result = ""

## Stop

Do not complete the retrieval task today. Session 5 is complete only after the capstone tests pass and you can explain each test without reading its comments.